# Oxta theses validation — Colab runner

Runs the rigorous validation harness for the two architectural theses.
Each test isolates ONE claim with a controlled baseline; results land in
Drive so you can compare across sessions.

**Hardware needs:**
- Quick mode: ~30 min on a T4 (free Colab is enough)
- Full mode: ~3-4 h on a T4 (one session)

**Important:** this is a separate notebook from `train_v11.ipynb`. It does
NOT touch the NSOS production code — it spins up small PyTorch models to
validate the techniques in isolation. Run it in a different Colab session
than your training, or stop training first.

Source code lives at `research/theses_validation/` in the repo.

## Step 1 — Mount Drive + clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/vitorGgC569/reimagined.git'
REPO_ROOT = Path('/content/reimagined')
DRIVE_ROOT = Path('/content/drive/MyDrive/nsos_v11/research_results')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# Optional auth (same pattern as train_v11.ipynb)
GH_TOKEN = None
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN')
except Exception:
    pass

def gh_url():
    if GH_TOKEN and 'github.com' in REPO_URL:
        return REPO_URL.replace('https://', f'https://oauth2:{GH_TOKEN}@')
    return REPO_URL

if REPO_ROOT.exists():
    print(f'[git] pulling latest in {REPO_ROOT}')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)
else:
    print(f'[git] cloning {REPO_URL}')
    subprocess.run(['git', 'clone', '--depth', '1', gh_url(), str(REPO_ROOT)], check=True)

subprocess.run(['git', '-C', str(REPO_ROOT), 'log', '-1', '--oneline'], check=True)

os.environ['REPO_ROOT'] = str(REPO_ROOT)
os.environ['DRIVE_RESULTS'] = str(DRIVE_ROOT)

## Step 2 — Install dependencies

`mamba_ssm` and `causal_conv1d` need a CUDA build environment — they take
a few minutes on first install. If they fail, the LrcSSM-vs-Mamba2 test
will skip gracefully.

In [ ]:
import sys, subprocess

req_path = REPO_ROOT / 'research/theses_validation/requirements.txt'
print(f'[deps] installing from {req_path}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)], check=False)

# Sanity check what's available
import importlib
for pkg in ['torch', 'datasets', 'numpy', 'tqdm', 'ncps', 'gymnasium']:
    try:
        m = importlib.import_module(pkg)
        print(f'  {pkg:<15} ok  ({getattr(m, "__version__", "?")})')
    except ImportError as e:
        print(f'  {pkg:<15} MISSING ({e})')

# mamba_ssm is platform-specific
try:
    from mamba_ssm import Mamba2
    print('  mamba_ssm       ok (Mamba2 importable)')
except Exception as e:
    print(f'  mamba_ssm       SKIP — LrcSSM-vs-Mamba2 will run LrcSSM only ({type(e).__name__})')

## Step 3 — Run the suite

Pick one of the two modes below.

- **Quick** (~30 min on T4) — reduced step counts; useful for a first pass and to verify everything runs.
- **Full** (~3-4 h on T4) — production step counts that match the README's reported numbers.

Results land in your Drive at `MyDrive/nsos_v11/research_results/<timestamp>/`.

In [ ]:
# Quick mode — ~30 min on T4
import subprocess, sys, time
from datetime import datetime

stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
results_dir = DRIVE_ROOT / f'{stamp}_quick'

cmd = [
    sys.executable, '-u',
    str(REPO_ROOT / 'research/theses_validation/run_all.py'),
    '--quick',
    '--seed', '42',
    '--results_dir', str(results_dir),
]
print(f'[suite] {" ".join(str(c) for c in cmd)}\n')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          bufsize=1, text=True)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    print('\n[suite] interrupted — sending SIGTERM')
    proc.terminate()
    raise
print(f'\n[suite] exit code: {proc.returncode}')
print(f'[suite] results: {results_dir}')

### OR — Full mode (only if you have a few hours)

In [ ]:
# Full mode — ~3-4 h on T4. Comment-out the quick cell above first.
import subprocess, sys
from datetime import datetime

stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
results_dir = DRIVE_ROOT / f'{stamp}_full'

cmd = [
    sys.executable, '-u',
    str(REPO_ROOT / 'research/theses_validation/run_all.py'),
    '--full',
    '--seed', '42',
    '--results_dir', str(results_dir),
]
print(f'[suite] {" ".join(str(c) for c in cmd)}\n')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          bufsize=1, text=True)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
except KeyboardInterrupt:
    print('\n[suite] interrupted')
    proc.terminate()
    raise
print(f'\n[suite] exit code: {proc.returncode}')
print(f'[suite] results: {results_dir}')

## Step 4 — Inspect results

List the JSON files that landed in Drive. Each one is a single test's output.

In [ ]:
import json
from pathlib import Path

print(f'[results] looking in {DRIVE_ROOT}')
for run_dir in sorted(DRIVE_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue
    print(f'\n  {run_dir.name}/')
    for json_path in sorted(run_dir.glob('*.json')):
        size_kb = json_path.stat().st_size / 1024
        print(f'    {json_path.name:<30}  {size_kb:>8.1f} KB')

## Step 5 — Single test (optional)

Re-run one specific test with custom args without going through `run_all.py`.

In [ ]:
# Example: run only the Decoupled STE comparison with extra steps.
import subprocess, sys

out = REPO_ROOT / 'research/theses_validation/results/single_decoupled_ste.json'
subprocess.run([
    sys.executable, '-u',
    str(REPO_ROOT / 'research/theses_validation/tests/quantization.py'),
    '--test', 'decoupled_ste_vs_ste',
    '--seed', '42',
    '--steps', '2500',
    '--out', str(out),
], check=False)